# ChaoSSM — Phase 21: does trained chaos help as the dependency gets longer?

Addresses **Reviewer 1, comment 4**: the trained-setting result (Section 3.4) uses short synthetic
tasks; *verify whether "chaotic overlap after training brings no advantage" is only an artifact of
toy (short) sequences* by adding a task with a longer dependency.

**Why a delay-recall sweep, not longer copy.** The trained chaotic arms already floor on the copy
task at length 12 (≈0.10–0.16 vs the GRU's 0.995); lengthening copy would push every arm except
the GRU to the chance floor and the chaos-vs-ablation comparison would lose all power — the same
low-power trap the NARMA benchmark fell into in Phase 18. Instead we use a **single-token
delay-recall task with a tunable memory horizon D**: present one content token, then D filler
steps, then a query; the model must emit the original token. Accuracy degrades *gradually* with D,
so the comparison keeps power across the whole range.

**The test.** We train the full gated model end-to-end (the Section 3.4 protocol) and sweep
D ∈ {4, 12, 24, 48}. The key comparison is **chaossm\_free (chaos) vs chaossm\_free\_periodic (its
periodic-regime ablation)**: if chaos conferred a memory advantage that only appears at long
dependency, chaossm\_free would sustain accuracy to larger D than its periodic twin. The paper's
mechanism (trained models contract on-task, chaos is suppressed) predicts it will not — the two
degrade together at every horizon, both dominated by the GRU. We also measure the driven
(conditional) Lyapunov exponent of the trained chaotic arms at the longest horizon to confirm they
contract on-task even there.

Runtime: ~1.5–2.5 h on a T4 (80 trained runs). Resumable — rerun the run cell after a disconnect.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, field, asdict
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM",phase="phase21_trained_longdep"):
    base=None
    try:
        from google.colab import drive
        try: drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}); retrying force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive NOT mounted ({type(e).__name__}). Results -> {base}")
        print("[WARNING] That is local disk, lost on runtime recycle. Run the rescue cell or download.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=600,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)

torch 2.11.0+cu130 | device: cuda
gpu: NVIDIA RTX PRO 6000 Blackwell Server Edition
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase21_trained_longdep


## Configuration — the trained-setting model (Section 3.4), plus the delay sweep

In [2]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=48; d_model:int=48; vocab:int=16; n_layers:int=2; mlp_mult:int=4
    steps:int=4000; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1; grad_clip:float=1.0
    gamma_init:float=1.0
    r_cap_chaotic:float=4.0; r_cap_periodic:float=3.4     # periodic < 3.57 onset (paper value)
    arms:tuple=("gru","linear","chaossm_free","chaossm_free_periodic")
    delays:tuple=(4,12,24,48)                              # memory-horizon sweep
    seeds:int=5
    eval_n:int=2000; eval_seed:int=12345
    lr:dict=field(default_factory=lambda: dict(gru=3e-3, linear=1e-4,
                   chaossm_free=1e-4, chaossm_free_periodic=1e-4))
CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=1; CFG.eval_n=40; CFG.delays=(4,12); CFG.arms=("gru","chaossm_free","chaossm_free_periodic")
    print("*** SMOKE TEST — pipeline check only, results meaningless ***")

def arm_is_chaossm(arm): return arm.startswith("chaossm")
def arm_is_periodic(arm): return arm.endswith("_periodic")
def arm_uses_corridor(arm): return False      # no corridor arms in this phase
print("arms:",CFG.arms,"| delays:",CFG.delays,"| seeds:",CFG.seeds)
print("planned runs =",len(CFG.arms)*len(CFG.delays)*CFG.seeds)

arms: ('gru', 'linear', 'chaossm_free', 'chaossm_free_periodic') | delays: (4, 12, 24, 48) | seeds: 5
planned runs = 80


## Cell, maps, gated block, model — verbatim from the trained battery (Phase 4)

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    '''Logistic-style map with a hard cap on r. r_cap=4.0 allows chaos; r_cap=3.4 forbids it.'''
    def __init__(self, N, r_cap, gamma_init=1.0):
        super().__init__()
        self.r_cap = float(r_cap)
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return self.r_cap * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, arm):
        super().__init__()
        self.N = cfg.state_dim
        self.is_chaossm = arm_is_chaossm(arm)
        if self.is_chaossm:
            r_cap = cfg.r_cap_periodic if arm_is_periodic(arm) else cfg.r_cap_chaotic
            self.phi = ChaoticMap(cfg.state_dim, r_cap, cfg.gamma_init)
        else:
            self.phi = LinearMap(cfg.state_dim)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.state_dim) / math.sqrt(cfg.state_dim))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.state_dim) / math.sqrt(cfg.state_dim))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.is_chaossm else torch.tanh(raw)
    def init_state(self, b, dev, dtype):
        v = 0.5 if self.is_chaossm else 0.0
        return torch.full((b, self.N), v, device=dev, dtype=dtype)

class GRUArmCell(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.N = cfg.state_dim
        self.cell = nn.GRUCell(cfg.state_dim, cfg.state_dim)
    def step(self, h, x): return self.cell(x, h)
    def init_state(self, b, dev, dtype): return torch.zeros(b, self.N, device=dev, dtype=dtype)

def make_cell(cfg, arm):
    return GRUArmCell(cfg) if arm == "gru" else ChaoSSMCell(cfg, arm)

In [4]:
class Block(nn.Module):
    def __init__(self, cfg, arm):
        super().__init__(); self.arm = arm
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.cell = make_cell(cfg, arm)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        hidden = cfg.d_model * cfg.mlp_mult
        self.mlp = nn.Sequential(nn.Linear(cfg.d_model, hidden), nn.GELU(),
                                 nn.Linear(hidden, cfg.d_model))
    def step(self, h, x):
        u = self.ln1(x); h = self.cell.step(h, u); y = x + h
        y = y + self.mlp(self.ln2(y)); return h, y
    def init_state(self, b, dev, dtype): return self.cell.init_state(b, dev, dtype)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, arm, vocab_size, readout_size):
        super().__init__(); self.cfg = cfg; self.arm = arm
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, arm) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.readout = nn.Linear(cfg.d_model, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); dtype = x.dtype
        states = [blk.init_state(B, tok.device, dtype) for blk in self.blocks]
        outs = []
        trajs = [[] for _ in self.blocks] if return_traj else None
        xseqs = [[] for _ in self.blocks] if return_traj else None
        for t in range(T):
            xt = x[:, t]
            for li, blk in enumerate(self.blocks):
                h_in = xt
                states[li], xt = blk.step(states[li], xt)
                if return_traj and arm_needs_traj(blk.arm):
                    trajs[li].append(states[li]); xseqs[li].append(blk.ln1(h_in))
            outs.append(self.readout(self.ln_f(xt)))
        logits = torch.stack(outs, 1)
        if not return_traj: return logits
        traj_out = {li: (torch.stack(trajs[li],1), torch.stack(xseqs[li],1))
                    for li in range(len(self.blocks)) if trajs[li]}
        return logits, traj_out

## Delay-recall task

`[content] + [PAD]×D + [QCUE] + [SEP] + [PAD]` → emit `content` at the final position. The content
token is drawn from the content vocabulary; the model must carry it across D filler steps. Chance =
1/V. The dependency length is exactly D (+3 structural tokens).

In [5]:
V=CFG.vocab
SEP, PAD, QCUE = V, V+1, V+2
VOCAB_SIZE=V+3; READOUT_SIZE=V+3
CHANCE=1.0/V

def make_delay_recall(cfg, D, batch, device, gen):
    content=torch.randint(0,V,(batch,1),device=device,generator=gen)        # the token to recall
    fillers=torch.full((batch,D),PAD,device=device)
    cue=torch.full((batch,1),QCUE,device=device); sep=torch.full((batch,1),SEP,device=device)
    tail=torch.full((batch,1),PAD,device=device)
    inp=torch.cat([content,fillers,cue,sep,tail],1)                         # (batch, D+4)
    T=inp.shape[1]
    tgt=torch.full((batch,T),-100,device=device); tgt[:,-1]=content[:,0]    # answer at last pos
    return inp,tgt

def build_eval(cfg,D):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(cfg.eval_seed)
    return make_delay_recall(cfg,D,cfg.eval_n,DEVICE,g)

@torch.no_grad()
def eval_acc(model,evalset,chunk=256):
    inp,tgt=evalset; model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        lg=model(inp[i:i+chunk]); tg=tgt[i:i+chunk]; m=tg!=-100; p=lg.argmax(-1)
        c+=((p==tg)&m).float().sum().item(); tot+=m.float().sum().item()
    model.train(); return c/max(tot,1)

# sanity: shapes and chance
for D in CFG.delays:
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(0)
    inp,tgt=make_delay_recall(CFG,D,4,DEVICE,g)
    print(f"D={D:2d}: seq_len={inp.shape[1]} (content + {D} fillers + cue + sep + answer-slot), chance={CHANCE:.4f}")

D= 4: seq_len=8 (content + 4 fillers + cue + sep + answer-slot), chance=0.0625
D=12: seq_len=16 (content + 12 fillers + cue + sep + answer-slot), chance=0.0625
D=24: seq_len=28 (content + 24 fillers + cue + sep + answer-slot), chance=0.0625
D=48: seq_len=52 (content + 48 fillers + cue + sep + answer-slot), chance=0.0625


## Driven (conditional) Lyapunov of the trained chaotic cell

In [6]:
from torch.func import jacrev
@torch.no_grad()
def conditional_lyapunov_trained(model, D, seed=0, n_warm=40):
    """Maximal driven Lyapunov exponent of the trained gated chaotic cell (layer 0) along a
    delay-recall input. Confirms whether the trained model contracts on-task. Returns NaN for
    non-chaossm arms."""
    blk=model.blocks[0]; cell=blk.cell
    if not getattr(cell,"is_chaossm",False): return float("nan")
    dev=next(model.parameters()).device; N=CFG.state_dim
    g=torch.Generator(device=dev); g.manual_seed(seed)
    inp,_=make_delay_recall(CFG,D,1,dev,g)
    emb=model.emb(inp)[0]                                    # (T, d_model)
    h=cell.init_state(1,dev,torch.float32).squeeze(0)
    Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev); T=emb.shape[0]; n_acc=0
    def driven(z, xvec):
        gte=torch.sigmoid(xvec @ cell.W_g.T + cell.b_g)
        raw=gte*cell.phi(z @ cell.A.T) + (1-gte)*z + xvec @ cell.B.T
        return triangle_fold(raw)
    for t in range(T):
        xt=blk.ln1(emb[t])
        J=torch.autograd.functional.jacobian(lambda z: driven(z, xt), h)
        h=driven(h, xt)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=n_warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12); n_acc+=1
    return float(torch.sort(logsum/max(n_acc,1),descending=True)[0][0])

## Train one arm at one delay

In [7]:
def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

def train_one(arm, D, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    model=ChaoSSMLM(cfg,arm,VOCAB_SIZE,READOUT_SIZE).to(DEVICE)
    opt=torch.optim.AdamW(model.parameters(),lr=cfg.lr[arm],betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    evalset=build_eval(cfg,D); div=False
    for step in range(cfg.steps):
        inp,tgt=make_delay_recall(cfg,D,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1),ignore_index=-100)
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip).item(); opt.step(); sched.step()
        if not (math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=float("nan") if div else eval_acc(model,evalset)
    lam=conditional_lyapunov_trained(model,D,seed=seed) if (arm_is_chaossm(arm) and not div) else float("nan")
    return dict(arm=arm,delay=D,seed=seed,diverged=div,held_out_acc=acc,
                conditional_lambda=lam,n_params=sum(p.numel() for p in model.parameters()))

## Run — resumable, checkpoints after every run

In [8]:
FRESH_START=False
CKPT=os.path.join(RESULTS,"phase21_trained_longdep.csv")
COLS=["arm","delay","seed","diverged","held_out_acc","conditional_lambda","n_params"]
if FRESH_START and os.path.exists(CKPT): os.replace(CKPT,CKPT+".bak")
if os.path.exists(CKPT):
    rows=pd.read_csv(CKPT).to_dict("records")
    done={(r["arm"],int(r["delay"]),int(r["seed"])) for r in rows}
    print(f"[resume] {len(rows)} runs done")
else: rows=[]; done=set(); print("[resume] fresh start")
def _flush(): pd.DataFrame(rows)[COLS].to_csv(CKPT,index=False)

plan=[(a,D,s) for a in CFG.arms for D in CFG.delays for s in range(CFG.seeds)]
todo=[k for k in plan if k not in done]
print(f"{len(plan)} planned, {len(plan)-len(todo)} done, {len(todo)} to run\n")
t0=time.time(); first=None
for arm,D,sd in todo:
    rt0=time.time(); r=train_one(arm,D,sd,CFG); rows.append(r); _flush()
    lt="" if math.isnan(r["conditional_lambda"]) else f" cond_lam={r['conditional_lambda']:+.3f}"
    print(f"[P21] {arm:24s} D={D:2d} s={sd} acc={r['held_out_acc']:.4f}{lt}"+(" DIVERGED" if r["diverged"] else ""))
    if first is None:
        first=time.time()-rt0; print(f"    [timing] ~{first:.0f}s/run -> ~{first*len(todo)/60:.0f} min remaining")
RES=pd.DataFrame(rows)[COLS]; _flush()
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min | runs: {len(RES)}")
print("n_params per arm:",RES.groupby("arm").n_params.first().to_dict())

[resume] 60 runs done
80 planned, 60 done, 20 to run

[P21] chaossm_free_periodic    D= 4 s=0 acc=1.0000 cond_lam=+0.000
    [timing] ~50s/run -> ~17 min remaining
[P21] chaossm_free_periodic    D= 4 s=1 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D= 4 s=2 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D= 4 s=3 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D= 4 s=4 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=12 s=0 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=12 s=1 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=12 s=2 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=12 s=3 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=12 s=4 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=24 s=0 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=24 s=1 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D=24 s=2 acc=1.0000 cond_lam=+0.000
[P21] chaossm_free_periodic    D

## Analysis — does chaos extend the memory horizon beyond its periodic ablation?

In [9]:
piv=RES.groupby(["arm","delay"]).held_out_acc.mean().unstack("delay").reindex(CFG.arms)
sd =RES.groupby(["arm","delay"]).held_out_acc.std().unstack("delay").reindex(CFG.arms)
savetable(RES.groupby(["arm","delay"]).held_out_acc.agg(["mean","std"]).reset_index(),"phase21_summary.csv")
print("held-out accuracy by arm x delay (chance = %.3f):"%CHANCE); print(piv.round(4).to_string()); print()
# THE key comparison: chaos vs its periodic ablation, at every delay
if "chaossm_free" in piv.index and "chaossm_free_periodic" in piv.index:
    print("chaos - periodic-ablation at each delay (should be ~0 at every horizon):")
    for D in CFG.delays:
        c=RES[(RES.arm=="chaossm_free")&(RES.delay==D)].held_out_acc.values
        p=RES[(RES.arm=="chaossm_free_periodic")&(RES.delay==D)].held_out_acc.values
        diff=np.nanmean(c)-np.nanmean(p)
        rng=np.random.default_rng(1); b=rng.choice(c-p,(20000,len(c))).mean(1)
        print(f"  D={D:2d}: chaos {np.nanmean(c):.4f} - periodic {np.nanmean(p):.4f} = {diff:+.4f} "
              f"CI[{np.percentile(b,2.5):+.4f},{np.percentile(b,97.5):+.4f}]")
# conditional lambda of the trained chaotic arms (contraction check)
print("\ntrained-model driven (conditional) lambda — negative = contracting on-task:")
for arm in CFG.arms:
    if arm_is_chaossm(arm):
        lam=RES[RES.arm==arm].conditional_lambda
        print(f"  {arm:24s} mean cond_lambda = {lam.mean():+.3f} (positive on {int((lam>0).sum())}/{lam.notna().sum()} runs)")
print("\nREADING: if chaos never beats its periodic ablation at any D, and the trained chaotic")
print("models contract (lambda<0) even at the longest horizon, the no-advantage conclusion is")
print("NOT a short-sequence artifact — it holds across the dependency range.")

saved /content/drive/My Drive/ChaoSSM/results/phase21_trained_longdep/phase21_summary.csv
held-out accuracy by arm x delay (chance = 0.062):
delay                   4       12      24      48
arm                                               
gru                    1.0  1.0000  1.0000  1.0000
linear                 1.0  0.9875  0.1428  0.0631
chaossm_free           1.0  1.0000  1.0000  0.2242
chaossm_free_periodic  1.0  1.0000  1.0000  0.8439

chaos - periodic-ablation at each delay (should be ~0 at every horizon):
  D= 4: chaos 1.0000 - periodic 1.0000 = +0.0000 CI[+0.0000,+0.0000]
  D=12: chaos 1.0000 - periodic 1.0000 = +0.0000 CI[+0.0000,+0.0000]
  D=24: chaos 1.0000 - periodic 1.0000 = +0.0000 CI[+0.0000,+0.0000]
  D=48: chaos 0.2242 - periodic 0.8439 = -0.6197 CI[-0.8390,-0.4112]

trained-model driven (conditional) lambda — negative = contracting on-task:
  chaossm_free             mean cond_lambda = +0.082 (positive on 5/20 runs)
  chaossm_free_periodic    mean cond_lambda = +0.

## Figure (600 dpi) — memory-horizon curves

In [10]:
COL={"gru":"#8456B0","linear":"#9E9E9E","chaossm_free":"#2E7D32","chaossm_free_periodic":"#B71C1C"}
LAB={"gru":"GRU","linear":"linear","chaossm_free":"chaotic (free)","chaossm_free_periodic":"periodic ablation"}
fig,ax=plt.subplots(figsize=(7.4,5.0))
for arm in CFG.arms:
    m=[RES[(RES.arm==arm)&(RES.delay==D)].held_out_acc.mean() for D in CFG.delays]
    s=[RES[(RES.arm==arm)&(RES.delay==D)].held_out_acc.std() for D in CFG.delays]
    ax.errorbar(CFG.delays,m,yerr=s,marker="o",ms=6,lw=2,capsize=4,color=COL.get(arm,"#333"),
                label=LAB.get(arm,arm))
ax.axhline(CHANCE,ls="--",color="#777",lw=1.2,label=f"chance ({CHANCE:.3f})")
ax.set_xlabel("memory horizon $D$ (filler steps between content and query)")
ax.set_ylabel("held-out recall accuracy")
ax.set_title("Trained setting: chaos does not extend the memory horizon\nbeyond its periodic ablation",fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=.25); ax.set_xticks(CFG.delays)
fig.tight_layout(); savefig(fig,"phase21_memory_horizon.png")

saved /content/drive/My Drive/ChaoSSM/results/phase21_trained_longdep/phase21_memory_horizon.png


## Rescue if Drive was not mounted

In [11]:
import shutil
if not RESULTS.startswith("/content/drive"):
    z=shutil.make_archive("/content/phase21_results","zip",RESULTS); print("archive:",z)
    try:
        from google.colab import files; files.download(z)
    except Exception as e: print("download",z,"manually:",e)
else: print("on Drive:",RESULTS)

on Drive: /content/drive/My Drive/ChaoSSM/results/phase21_trained_longdep
